Outlining a feedforward neural network without Pytorch, no TensorFlow
The point is to implement some of these structures to develop stronger intuitions and understanding of these larger paradigms

In [6]:
import math
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

In [7]:
class Linear:
    def __init__(self, in_features, out_features):
        self.W = np.random.randn(in_features, out_features) * 0.01
        self.b = np.zeros(out_features)

    def forward(self, X):
        self.X = X
        return X @ self.W + self.b

    def backward(self, dout):
        self.dW = self.X.T @ dout
        self.db = dout.sum(axis=0)
        return dout @ self.W.T

class ReLU:
    def forward(self, X):
        self.X = X
        return np.maximum(0, X)

    def backward(self, dout):
        return np.where(self.X > 0, 1, 0) * dout

class SoftmaxCrossEntropy:
    def forward(self, X, y):
        self.y = y
        self.batch_size = X.shape[0]
        exps = np.exp(X - np.max(X, axis=1, keepdims=True))
        self.probs = exps / np.sum(exps, axis=1, keepdims=True)

        class_probability = self.probs[np.arange(X.shape[0]), y]
        return np.mean(-1 * np.log(class_probability))

    
    def backward(self):
        one_hot = np.zeros((self.batch_size, 10))
        one_hot[np.arange(self.batch_size), self.y] = 1
        return (self.probs - one_hot) / self.batch_size

In [8]:
mnist = fetch_openml('mnist_784', version=1)
X, y = mnist.data.to_numpy(), mnist.target.astype(int).to_numpy()

Here we preprocess the data

In [9]:
print(X.shape)
print(y.shape)
print(X.min(), X.max())

(70000, 784)
(70000,)
0 255


In [10]:
X = X / 255.0

In [11]:
X_train, X_test = X[:60000], X[60000:]
y_train, y_test = y[:60000], y[60000:]

Now we actually start the training loop

In [12]:
epochs = 100
batch_size = 32
linear_1 = Linear(784, 128)
linear_2 = Linear(128, 10)
relu = ReLU()
softmaxCE = SoftmaxCrossEntropy()
lr = 0.05
for epoch in range(epochs):
    epoch_loss = 0
    for i in range(0, 60000, batch_size):
        # Slice up data into batch size
        X_slice = X_train[i:i + batch_size]
        y_slice = y_train[i:i + batch_size]

        # Forward
        forward_linear_result = linear_1.forward(X_slice)
        relu_forward = relu.forward(forward_linear_result)
        forward_linear_2_result = linear_2.forward(relu_forward)
        loss = softmaxCE.forward(forward_linear_2_result, y_slice)

        # Update Epoch_Loss
        epoch_loss += loss

        # Backward
        backward_softmaxCE = softmaxCE.backward()
        backward_linear_2_result = linear_2.backward(backward_softmaxCE)
        relu_backward = relu.backward(backward_linear_2_result)
        backward_linear_1_result = linear_1.backward(relu_backward)

        # Update weights
        linear_1.W -= lr * linear_1.dW
        linear_1.b -= lr * linear_1.db
        linear_2.W -= lr * linear_2.dW
        linear_2.b -= lr * linear_2.db
        
    print(f"Loss[{epoch}]: {epoch_loss}")







Loss[0]: 994.9224653032162
Loss[1]: 451.10363199946306
Loss[2]: 339.430220031433
Loss[3]: 272.6901917066993
Loss[4]: 228.47990970635573
Loss[5]: 196.57489316796963
Loss[6]: 172.2468795487377
Loss[7]: 152.9247337532994
Loss[8]: 137.32215190290634
Loss[9]: 124.22733090408299
Loss[10]: 113.0169745503325
Loss[11]: 103.2648921825842
Loss[12]: 94.7120351587116
Loss[13]: 87.21330238697058
Loss[14]: 80.50557101488563
Loss[15]: 74.49048259730486
Loss[16]: 68.98238543029633
Loss[17]: 63.97571778417178
Loss[18]: 59.44411123757646
Loss[19]: 55.27221817406585
Loss[20]: 51.409270771438635
Loss[21]: 47.92947382867388
Loss[22]: 44.69923627081577
Loss[23]: 41.75407837653653
Loss[24]: 39.02337575729586
Loss[25]: 36.54429904879579
Loss[26]: 34.2260926311049
Loss[27]: 32.10867688951042
Loss[28]: 30.18474851338004
Loss[29]: 28.376752555553274
Loss[30]: 26.713007891291202
Loss[31]: 25.168907834620022
Loss[32]: 23.747264847256034
Loss[33]: 22.446647743035523
Loss[34]: 21.165613331687897
Loss[35]: 20.08437614